# <font color='#2E86C1' size=6>**Data cleaning: diagnosis and leak-safe rules**</font>

**Workflow:** **01 Data cleaning** → [02 Customer table](02_customer_table.ipynb) → [03 Missing values](03_missing_values.ipynb) → [04 Outliers](04_outlier_detection.ipynb) → [05 Preprocessing recipe and models](05_preprocessing_and_models.ipynb) → [06 Feature engineering and selection](06_feature_engineering_selection.ipynb)

Diagnosis of `train.csv` / `test.csv` and the rules and evidence behind `src/` and `config.yaml`: column names, placeholders, schema, dates, duplicates, identities and consistency rules. The notebook ends by setting aside the locked test set and the temporal folds that every later notebook uses to compare options. The notebook imports the pipeline's own functions, so it always runs the same code as `python main.py`.

## Table of Contents <a class="anchor" id="toc"></a>
- [0. Imports and configuration](#imports)
- [1. Data import](#data-import)
- [2. Cleaning and validation](#cleaning)
- [3. Date validation](#dates)
- [4. Duplicates](#duplicates)
- [5. From diagnosis to pipeline rules](#rules)
- [6. Features, targets and the locked test set](#split)
- [7. Temporal cross-validation and the decision rule](#cv)
- [8. Consistency rules in the models](#consistency-models)
- [9. From this notebook to the pipeline](#to-pipeline)

# <font color='#2E86C1' size=6>**0. Imports and configuration**</font> <a class="anchor" id="imports"></a>

[Back to TOC](#toc)

In [1]:
import copy
import os
import sys
from pathlib import Path

# run from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml

from src.data import (
    load_data, report_placeholders, validate_schema, validate_customer_ids, standardize_columns, standardize_dtypes,
)
from src.cleaning import flag_invalid_values, apply_consistency_rules, prepare_table, drop_training_rows, fill_from_customer_history
from src.preprocessing import split_dev_test, task_preprocessing_config
from src.evaluate import select_scoring, temporal_folds, cross_validate_pipeline, paired_comparison
from src.pipeline import build_pipeline, prepare_task_data
from src.tasks import TASKS
import classification.preprocessing as classification_prep
import regression.preprocessing as regression_prep
import classification.model as classification_model
import regression.model as regression_model

# every rule and choice (column lists, placeholder value, seeds, ...) comes from config.yaml
with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
data_cfg, diag_cfg, prep_cfg = config["data"], config["diagnostics"], config["preprocessing"]

# each task's metrics (config.yaml -> <task>.metrics) as sklearn scoring strings
SCORING = {task: select_scoring(TASKS[task].evaluate.SCORING, config[task]["metrics"]) for task in TASKS}


# <font color='#2E86C1' size=6>**1. Data import**</font> <a class="anchor" id="data-import"></a>

[Back to TOC](#toc)

In [2]:
# placeholder tokens ("?", "N/A", blank, ...) become NaN on load -- see section 2.2
tokens = diag_cfg["placeholder_tokens"]
train_raw = load_data(data_cfg["train_path"], tokens)
test_raw = load_data(data_cfg["test_path"], tokens)
clientes_raw = load_data(data_cfg["clientes_path"], tokens)

### 1.1 Column-name standardisation

The CSV files are stored as `train_raw`, `test_raw`, and `clientes_raw` for traceability. Standardised copies with lowercase English `snake_case` names become the working datasets used by every subsequent cleaning step. `standardize_columns()` (in `src/data.py`) uses the explicit names in `config.yaml` → `diagnostics.column_renames` and a generic snake_case rule for the rest, and raises an error if two columns end up with the same name.

In [3]:
train = standardize_columns(train_raw, diag_cfg["column_renames"])
test = standardize_columns(test_raw, diag_cfg["column_renames"])
clientes = standardize_columns(clientes_raw, diag_cfg["column_renames"])

column_changes = pd.DataFrame({
    "original": train_raw.columns,
    "standardized": train.columns,
})

display(column_changes.query("original != standardized"))
print("All standardized schemas have unique column names.")


,original,standardized
0,Total a Pagar,total_payable
1,Total Bruto,gross_total
2,Total Liquido,net_total
3,Total Imp_,total_tax
4,Total Portes,shipping_cost
5,Total Desc_ Global,global_discount_total
6,Total Desc_ Linha,line_discount_total
7,Custo,cost
8,N_ Detalhes,line_item_count
38,ID,id


All standardized schemas have unique column names.


**Conclusion:** Three files loaded: train.csv (6,534 purchases, both targets), test.csv (986 purchases, no targets) and clientes.csv (1,143 customers). 10 columns are renamed (the 9 Portuguese names and `ID`), and every name is unique. All later steps use these names.

# <font color='#2E86C1' size=6>**2. Cleaning and validation**</font> <a class="anchor" id="cleaning"></a>

[Back to TOC](#toc)

### 2.1 Almost-empty rows

Exploratory analysis found 65 rows where only `id`, `customer_id`, and the classification target are present. We reproduce that pattern using all remaining columns rather than a manually selected subset (`config.yaml` → `diagnostics.almost_empty_preserved_columns`). This cell only **diagnoses** them; they are removed from training in section 2.4.

In [4]:
preserved_columns = diag_cfg["almost_empty_preserved_columns"]
almost_empty_columns = [
    column for column in train.columns
    if column not in preserved_columns
]
missing_pattern = train[almost_empty_columns].isna().all(axis=1)

affected = train.loc[missing_pattern, ["id", "customer_id"]].copy()

# Check whether each customer_id exists in clientes.
customer_id_col = "customer_id"

affected["exists_in_clientes"] = affected["customer_id"].isin(
    clientes[customer_id_col]
)

# Display the results
print("Affected rows:", len(affected))
print("\nCross-check results:")
print(affected["exists_in_clientes"].value_counts(dropna=False))

print("\nCustomers not found in clientes:")
print(
    affected.loc[~affected["exists_in_clientes"]]
    .sort_values("customer_id")
)


Affected rows: 65

Cross-check results:
exists_in_clientes
True    65
Name: count, dtype: int64

Customers not found in clientes:
Empty DataFrame
Columns: [id, customer_id, exists_in_clientes]
Index: []


### 2.2 Placeholders and invalid numeric values

**Placeholder tokens.** `report_placeholders()` reads each raw file as plain text, before pandas converts anything, and counts every token in `config.yaml` → `diagnostics.placeholder_tokens` (blank, `-`, `?`, `N/A`, `null`, `unknown`, …). It also counts the numeric placeholder `1,000,000,000`:

In [5]:
placeholder_report = pd.concat(
    {name: report_placeholders(data_cfg[f"{name}_path"], diag_cfg) for name in ["train", "test", "clientes"]},
    names=["file"],
).reset_index(level=0)
display(placeholder_report.groupby(["file", "placeholder", "treatment"])["count"].agg(["sum", "count"])
        .rename(columns={"sum": "occurrences", "count": "columns"}))
display(placeholder_report.query("file == 'train'").sort_values("count", ascending=False).head(10))

occurrences  \
file     placeholder   treatment                                                 
clientes ''            NaN on load (load_data)                            3385   
test     ''            NaN on load (load_data)                             489   
         1,000,000,000 NaN in clean_dataset (flag_invalid_values)            4   
train    ''            NaN on load (load_data)                           12927   
         1,000,000,000 NaN in clean_dataset (flag_invalid_values)           31   

                                                                   columns  
file     placeholder   treatment                                            
clientes ''            NaN on load (load_data)                           9  
test     ''            NaN on load (load_data)                          12  
         1,000,000,000 NaN in clean_dataset (flag_invalid_values)        2  
train    ''            NaN on load (load_data)                          37  
         1,000,000,000 NaN in clean_dataset (flag_invalid_values)        2

,file,column,placeholder,count,treatment
10,train,spend_next_90d,'',3109,NaN on load (load_data)
21,train,prior_std_gap_days,'',1478,NaN on load (load_data)
20,train,prior_mean_gap_days,'',1308,NaN on load (load_data)
17,train,prior_std_purchase_value,'',1113,NaN on load (load_data)
15,train,prior_mean_purchase_value,'',862,NaN on load (load_data)
11,train,days_since_previous_purchase,'',861,NaN on load (load_data)
18,train,prior_min_purchase_value,'',668,NaN on load (load_data)
19,train,prior_max_purchase_value,'',668,NaN on load (load_data)
34,train,transport_method,'',261,NaN on load (load_data)
29,train,spend_365d,'',260,NaN on load (load_data)


- **Blank cells are the only text placeholder in the three files** (12,927 in train.csv). No `?`, `N/A`, `-` or `unknown` appears. `load_data()` still turns every listed token into `NaN`, so a new file that uses `?` can't make a numeric column load as text.
- **The numeric placeholder `1,000,000,000`** appears 31 times in train.csv and 4 times in test.csv, always in `prior_mean_purchase_value` and `spend_365d`. It becomes `NaN` in `clean_dataset()` and is then recovered from identities (5.1).
- **Negative amounts, counts and time intervals** are impossible. They become missing values too (`config.yaml` → `diagnostics.non_negative_columns`, `placeholder_values`, `placeholder_columns`). Section 5.1 then recovers most of them exactly from identities between columns. No rows are removed.

In [6]:
non_negative_columns = diag_cfg["non_negative_columns"]
placeholder_columns = diag_cfg["placeholder_columns"]
placeholder_values = diag_cfg["placeholder_values"]

invalid_value_report = []
for dataset_name, df in {"train": train, "test": test}.items():
    for column in non_negative_columns:
        count = int(df[column].lt(0).sum())
        if count:
            invalid_value_report.append([dataset_name, column, "negative", count])

    for column in placeholder_columns:
        count = int(df[column].isin(placeholder_values).sum())
        if count:
            invalid_value_report.append([dataset_name, column, "placeholder", count])

invalid_value_report = pd.DataFrame(
    invalid_value_report, columns=["dataset", "column", "issue", "count"]
)
display(invalid_value_report)


,dataset,column,issue,count
0,train,days_since_previous_purchase,negative,16
1,train,prior_mean_purchase_value,negative,16
2,train,prior_mean_gap_days,negative,16
3,train,spend_365d,negative,15
4,train,prior_mean_purchase_value,placeholder,16
5,train,spend_365d,placeholder,15
6,test,days_since_previous_purchase,negative,2
7,test,prior_mean_purchase_value,negative,2
8,test,prior_mean_gap_days,negative,2
9,test,spend_365d,negative,2


In [7]:
train_clean, test_clean = train.copy(), test.copy()
flag_invalid_values(train_clean, diag_cfg)      # in place -- the same function clean_dataset() uses
flag_invalid_values(test_clean, diag_cfg)

for name, df in {"train": train_clean, "test": test_clean}.items():
    negatives_left = int(df[non_negative_columns].lt(0).sum().sum())
    placeholders_left = int(df[placeholder_columns].isin(placeholder_values).sum().sum())
    print(f"{name}: {negatives_left} negative values and {placeholders_left} placeholders left; {len(df)} rows (none removed)")

train: 0 negative values and 0 placeholders left; 6534 rows (none removed)
test: 0 negative values and 0 placeholders left; 986 rows (none removed)


### 2.3 Categorical text values

We compare categories before and after trimming whitespace and standardising case. No case or spacing variants are found, so the original category labels are kept.


In [8]:
categorical_columns = prep_cfg["categorical_features"]

category_text_report = []
for column in categorical_columns:
    values = pd.concat([train_clean[column], test_clean[column]]).dropna().astype(str)
    normalized = values.str.strip().str.casefold()
    category_text_report.append({
        "column": column,
        "values_with_whitespace": int(values.ne(values.str.strip()).sum()),
        "case_or_spacing_variants": values.nunique() - normalized.nunique(),
    })

category_text_report = pd.DataFrame(category_text_report)
display(category_text_report)
assert category_text_report[["values_with_whitespace", "case_or_spacing_variants"]].eq(0).all().all()


,column,values_with_whitespace,case_or_spacing_variants
0,payment_type,0,0
1,salesperson,0,0
2,commercial_zone,0,0
3,transport_method,0,0
4,warehouse,0,0
5,document_series,0,0


### 2.4 Training-only row removal

`drop_training_rows()` removes the 65 almost-empty rows found in 2.1, plus exact duplicates (ignoring `id`) and repeated ids. It runs on the **labelled training file only**: every test.csv id needs a prediction, so test rows are never removed.

In [9]:
train_clean, row_report = drop_training_rows(train_clean, diag_cfg, data_cfg["id_column"])
display(row_report)
print("Training rows after removal:", len(train_clean), "| test rows:", len(test_clean), "(test.csv is never filtered)")

,rule,rows_removed
0,almost-empty rows,65
1,exact duplicates (ignoring id),0
2,repeated ids,0


Training rows after removal: 6469 | test rows: 986 (test.csv is never filtered)


### 2.5 Data-type standardisation

Counts and identifiers use compact nullable integer types, category labels use `category`, and the binary target uses `Int8`. Monetary values and calculated statistics remain `float64` so that decimals are preserved. The rules live in `config.yaml` → `dtypes` and are also applied by `prepare_table()`.

In [10]:
train_memory_before = train_clean.memory_usage(deep=True).sum()
test_memory_before = test_clean.memory_usage(deep=True).sum()
train_types_before = train_clean.dtypes.astype(str)
test_types_before = test_clean.dtypes.astype(str)

train_clean = standardize_dtypes(train_clean, config["dtypes"])
test_clean = standardize_dtypes(test_clean, config["dtypes"])

type_changes = pd.DataFrame({
    "train_before": train_types_before,
    "train_after": train_clean.dtypes.astype(str),
    "test_before": test_types_before,
    "test_after": test_clean.dtypes.astype(str),
})
changed = (
    type_changes["train_before"].ne(type_changes["train_after"])
    | type_changes["test_before"].ne(type_changes["test_after"])
)
display(type_changes.loc[changed])
print(f"Train memory: {train_memory_before / 1024**2:.2f} MB -> {train_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Test memory: {test_memory_before / 1024**2:.2f} MB -> {test_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

,train_before,train_after,test_before,test_after
commercial_zone,str,category,str,category
customer_id,int64,Int32,int64,Int32
customer_tenure_days,float64,Int16,float64,Int16
document_series,str,category,str,category
id,int64,Int32,int64,Int32
invoice_count_current_purchase,float64,Int16,float64,Int16
line_item_count,float64,Int16,float64,Int16
payment_type,str,category,str,category
prior_purchase_count,float64,Int16,float64,Int16
purchase_count_180d,float64,Int16,float64,Int16


Train memory: 4.64 MB -> 1.77 MB
Test memory: 0.68 MB -> 0.25 MB


### 2.6 Train/test feature compatibility

The two targets are expected to exist only in `train`. For every predictor shared by train and test, the data type must have the same broad meaning (numeric or text). Exact pandas dtypes may differ because missing numeric values are stored as floats.


In [11]:
target_columns = ["repeat_purchase_90d", "spend_next_90d"]
train_feature_columns = set(train_clean.columns) - set(target_columns)
test_feature_columns = set(test_clean.columns)

print("Features missing from test:", sorted(train_feature_columns - test_feature_columns))
print("Unexpected features in test:", sorted(test_feature_columns - train_feature_columns))

def dtype_family(series):
    return "numeric" if pd.api.types.is_numeric_dtype(series) else "text"

type_report = pd.DataFrame({
    "train_dtype": train_clean.dtypes,
    "test_dtype": test_clean.dtypes,
}).loc[sorted(train_feature_columns & test_feature_columns)]
type_report["train_family"] = [dtype_family(train_clean[column]) for column in type_report.index]
type_report["test_family"] = [dtype_family(test_clean[column]) for column in type_report.index]

display(type_report.query("train_dtype != test_dtype"))
assert train_feature_columns == test_feature_columns
assert type_report["train_family"].equals(type_report["test_family"])


Features missing from test: []
Unexpected features in test: []


,train_dtype,test_dtype,train_family,test_family
commercial_zone,category,category,text,text
document_series,category,category,text,text
payment_type,category,category,text,text
salesperson,category,category,text,text
transport_method,category,category,text,text
warehouse,category,category,text,text


### 2.7 Test categories unseen in training

Some valid test categories do not appear in training. They are not renamed: the later encoder must handle unknown categories safely.


In [12]:
unseen_category_report = []
for column in categorical_columns:
    train_categories = set(train_clean[column].dropna().astype(str))
    test_categories = set(test_clean[column].dropna().astype(str))
    unseen_in_train = sorted(test_categories - train_categories)
    unseen_category_report.append({
        "column": column,
        "test_categories_unseen_in_train": ", ".join(unseen_in_train) or "None",
    })

unseen_category_report = pd.DataFrame(unseen_category_report)
display(unseen_category_report.query("test_categories_unseen_in_train != 'None'"))


,column,test_categories_unseen_in_train
0,payment_type,"Maybe Tomorrow Payment 1, Maybe Tomorrow Payme..."
2,commercial_zone,"Decorative Dashboard Zone 1, Decorative Dashbo..."
3,transport_method,"Maybe Tomorrow Delivery 54, Maybe Tomorrow Del..."
4,warehouse,Final Excel Warehouse 2


### 2.8 Schema validation (expected types, valid ranges, allowed categories)

`validate_schema()` is the first step of `prepare_table()`, so every table goes through it: train.csv, test.csv and any new record. Its rules live in `config.yaml` → `schema`:
- **required columns:** an error if one is missing; extra columns are reported;
- **expected types:** numeric columns must be numeric (anything that can't be read as a number → `NaN`, counted), dates must parse, count columns must hold whole numbers;
- **valid ranges:** every non-negative column has min 0, plus explicit ranges (`purchase_date` from 2010 up to today, so a future date is caught, ≥ 1 line item and invoice, target 0/1, spend ≥ 0). Out-of-range values are counted here and turned into `NaN` by `clean_dataset()`;
- **allowed categories:** every label follows `"<fixed text> <number>"` (e.g. `Maybe Tomorrow Payment 12`). A label that doesn't match is not a category → `NaN`, counted. Labels that match but were never seen in training are allowed; the encoders handle them.

In [13]:
schema_reports = {}
for name, raw in {"train": train_raw, "test": test_raw}.items():
    validated, schema_reports[name] = validate_schema(
        standardize_columns(raw, diag_cfg["column_renames"]), config["schema"], diag_cfg["non_negative_columns"])
    assert len(validated) == len(raw)
schema_summary = pd.concat(schema_reports, names=["file"]).reset_index(level=0)
display(schema_summary.groupby(["file", "step"])["values"].sum().unstack("file").fillna(0).astype(int))
display(schema_summary.query("values > 0"))

file,test,train
step,,
schema: label not allowed -> NaN,0,0
schema: non-integer value,0,0
schema: outside valid range,8,63
schema: unparseable date,0,0


,file,step,column,values,rule
22,train,schema: outside valid range,days_since_previous_purchase,16,"[0, inf]"
26,train,schema: outside valid range,prior_mean_purchase_value,16,"[0, inf]"
30,train,schema: outside valid range,prior_mean_gap_days,16,"[0, inf]"
39,train,schema: outside valid range,spend_365d,15,"[0, inf]"
21,test,schema: outside valid range,days_since_previous_purchase,2,"[0, inf]"
25,test,schema: outside valid range,prior_mean_purchase_value,2,"[0, inf]"
29,test,schema: outside valid range,prior_mean_gap_days,2,"[0, inf]"
38,test,schema: outside valid range,spend_365d,2,"[0, inf]"


Both files have every required column with the expected type, every date parses, every count is a whole number, and **every category label is allowed**. The only out-of-range values are the 63 negatives in train.csv (8 in test.csv) already found in 2.2. They are counted here and become missing values in `clean_dataset()`.

**Conclusions:**
- **Almost-empty rows:** 65 training rows have nothing but ids and the label. They are removed from training only.
- **Placeholders:** the only text placeholder is the blank cell. The `1,000,000,000` placeholder (2 columns) and negative values (4 columns) become missing values, and section 5 recovers most of them.
- **Schema:** every required column, type, date and category label is valid (2.8). `validate_schema()` checks the same rules on every new table.
- **Category labels:** clean, with no case or spacing variants.
- **Spend target:** missing for all non-repeaters (structural), and *not at random* for 5.4% of repeaters, so it is never imputed (notebook 03, section 1).
- **Train vs test:** both share the same features with compatible types, but test.csv has categories never seen in training, so every encoder must handle unknown categories.

# <font color='#2E86C1' size=6>**3. Date validation**</font> <a class="anchor" id="dates"></a>

- 3.1 **Date validity:** Count missing and unparseable dates in both datasets. Avoid crashes when it contains no valid dates.
- 3.2 **Temporal coverage:** Report the minimum and maximum valid dates for each dataset, with consistent formatting.
- 3.3 **Train/test chronology:** Check whether the training period ends before the test period begins. Report inconsistencies as a warning.
- 3.4 **Temporal distribution:** Check whether dates are concentrated in particular periods and whether there are unexpected gaps, if the data is expected to cover a continuous timeline.

[Back to TOC](#toc)

In [14]:

def validate_purchase_dates(train, test):
    """Validate purchase dates and compare train/test time periods."""

    datasets = {"train": train.copy(), "test": test.copy()}
    date_col = "purchase_date"
    date_ranges = {}

    # Step 1: Check that both datasets contain the date column.
    for name, df in datasets.items():
        if date_col not in df.columns:
            raise KeyError(f"'{date_col}' is missing from {name}.")

    # Step 2: Parse dates and report invalid or missing values.
    for name, df in datasets.items():
        # Convert invalid date strings to NaT instead of raising errors.
        df[date_col] = pd.to_datetime(
            df[date_col],
            errors="coerce"
        )

        valid_dates = df[date_col].dropna()
        invalid_count = df[date_col].isna().sum()

        print(f"\n--- {name.upper()} DATE VALIDATION ---")
        print(f"Total observations: {len(df)}")
        print(f"Missing/unparseable dates: {invalid_count}")

        # Step 3: Calculate the date interval safely.
        if valid_dates.empty:
            print("No valid dates available.")
            date_ranges[name] = None
            continue

        earliest = valid_dates.min()
        latest = valid_dates.max()

        date_ranges[name] = (earliest, latest)

        print(f"Earliest date: {earliest}")
        print(f"Latest date:   {latest}")

        monthly_counts = valid_dates.dt.to_period("M").value_counts().sort_index()
        full_period = pd.period_range(
            earliest.to_period("M"), latest.to_period("M"), freq="M"
        )
        missing_months = full_period.difference(monthly_counts.index)
        print(f"Months without purchases: {len(missing_months)}")

    # Step 4: Compare the temporal coverage of both datasets.
    train_range = date_ranges["train"]
    test_range = date_ranges["test"]

    if train_range is None or test_range is None:
        print("\nCannot compare periods: at least one dataset has no valid dates.")
        return datasets["train"], datasets["test"]

    train_start, train_end = train_range
    test_start, test_end = test_range

    print("\n--- TRAIN/TEST CHRONOLOGY ---")

    if train_end < test_start:
        print("PASS: Training dates precede test dates.")
    elif test_end < train_start:
        print("WARNING: Test period precedes training period.")
    else:
        print("WARNING: Train/test date ranges overlap.")

        overlap_start = max(train_start, test_start)
        overlap_end = min(train_end, test_end)

        print(f"Overlapping interval: {overlap_start} to {overlap_end}")

    return datasets["train"], datasets["test"]


train_clean, test_clean = validate_purchase_dates(train_clean, test_clean)


--- TRAIN DATE VALIDATION ---
Total observations: 6469
Missing/unparseable dates: 0
Earliest date: 2015-07-11 00:00:00
Latest date:   2025-02-15 00:00:00
Months without purchases: 0

--- TEST DATE VALIDATION ---
Total observations: 986
Missing/unparseable dates: 0
Earliest date: 2025-05-24 00:00:00
Latest date:   2026-10-06 00:00:00
Months without purchases: 0

--- TRAIN/TEST CHRONOLOGY ---
PASS: Training dates precede test dates.


**Conclusions:** Every date parses, there are no months without purchases, and test.csv (2025-05 → 2026-10) starts after train.csv ends (2025-02). So the models must be validated on *later* purchases than they are trained on (section 6).

# <font color='#2E86C1' size=6>**4. Duplicates**</font> <a class="anchor" id="duplicates"></a>

- 4.1 Exact duplicate rows in each dataset, both on all columns and **ignoring `id`** (a unique `id` would otherwise hide every duplicate).
- 4.2 Repeated ids.
- 4.3 Test rows identical to a training row (shared columns except `id`), which would hint at leakage.
- 4.4 `customer_id` validation: present, a positive integer, known in clientes.csv, and a purchase history that never goes backwards for the same customer.

This section is **diagnostic only**. Training duplicates are removed by `drop_training_rows()` (section 2.4), and test rows are never removed, because every one of the 986 ids needs a prediction.

[Back to TOC](#toc)

In [15]:
def duplicate_diagnostics(train, test, id_column):
    """Counts duplicates in train and test and train/test overlaps. Removes nothing."""
    rows = []
    for name, df in {"train": train, "test": test}.items():
        non_id_columns = [c for c in df.columns if c != id_column]
        rows.append({
            "dataset": name,
            "rows": len(df),
            "exact duplicates (all columns)": int(df.duplicated().sum()),
            "duplicates ignoring id": int(df.duplicated(subset=non_id_columns).sum()),
            "repeated ids": int(df[id_column].duplicated().sum()),
        })

    shared = [c for c in train.columns if c in test.columns and c != id_column]
    train_hashes = set(pd.util.hash_pandas_object(train[shared], index=False))
    test_matches = int(pd.util.hash_pandas_object(test[shared], index=False).isin(train_hashes).sum())
    return pd.DataFrame(rows), test_matches

In [16]:
duplicate_table, test_rows_matching_train = duplicate_diagnostics(train_clean, test_clean, data_cfg["id_column"])
display(duplicate_table)
print("Test rows identical to a training row (shared columns except id):", test_rows_matching_train)

assert len(test_clean) == len(test_raw) == 986, "test.csv must keep every row: every id needs a prediction"

,dataset,rows,exact duplicates (all columns),duplicates ignoring id,repeated ids
0,train,6469,0,0,0
1,test,986,0,0,0


Test rows identical to a training row (shared columns except id): 0


**4.4 `customer_id` validation.** The same customer appears in many rows, which is expected: one row per purchase. What must hold is that the id is valid, and that each customer's history, taken in date order, only moves forward. The prior purchase count and the tenure can't decrease from one purchase to the next, and there can't be two purchases on the same day.

In [17]:
customer_checks = pd.concat({
    "train": validate_customer_ids(train_clean, clientes_raw, data_cfg),
    "test": validate_customer_ids(test_clean.assign(purchase_date=pd.to_datetime(test_clean["purchase_date"])), clientes_raw, data_cfg),
}, names=["file"]).reset_index(level=0).pivot(index="check", columns="file", values="rows")
display(customer_checks)
history_steps = (train_clean.assign(_d=pd.to_datetime(train_clean["purchase_date"])).sort_values(["customer_id", "_d"])
                 .groupby("customer_id")["prior_purchase_count"].diff().value_counts())
print("prior_purchase_count step between two consecutive purchases of a customer:", history_steps.to_dict())

file,test,train
check,,
customer_id not a positive integer,0,0
customer_id not in clientes.csv,0,0
customer_tenure_days decreases over a customer's purchases,0,0
missing customer_id,0,0
prior_purchase_count decreases over a customer's purchases,0,0
"same customer, same purchase date",0,0


prior_purchase_count step between two consecutive purchases of a customer: {np.int16(1): 5810, np.int16(2): 52}


**Conclusions:** There are no duplicates of any kind (all columns, ignoring `id`, or repeated ids), and no test row is identical to a training row. test.csv keeps all 986 rows. Every `customer_id` is a positive integer known in clientes.csv. Each customer's history is consistent: the prior purchase count grows by 1 (or 2) from one purchase to the next, the tenure never shrinks, and there are no same-day repeats. So the repeated customer ids are genuine repeat purchases, not duplicates.

# <font color='#2E86C1' size=6>**5. From diagnosis to pipeline rules**</font> <a class="anchor" id="rules"></a>


Every step is one of three kinds, and the kind decides where it may run:

| Kind | In this project | Learns from data? | Where it runs |
|---|---|---|---|
| **Stateless rule** | renames, negatives/placeholders → `NaN`, identity recomputation, structural fill, flags, redundant columns (`clean_dataset`) | No | train.csv **and** test.csv |
| **Training-data decision** | almost-empty rows, duplicates (`drop_training_rows`), regression rows (`regression.select_training_rows`) | No, but **removes rows** | labelled training file only, before the dev/test split |
| **Fitted step** | imputation, encoding, scaling (`build_preprocessor`) | **Yes** | inside the model `Pipeline`, fitted on training rows only |

[Back to TOC](#toc)

### 5.1 Values that can be recovered from identities

Several history columns are linked by exact definitions. Wherever one term is missing (originally, or after the negative/placeholder → `NaN` step), it can be recomputed **from the same row**. Nothing is learned from other rows, so this is a stateless rule and runs on test.csv too. The rules live in `config.yaml` → `diagnostics.recompute_rules`. Each one is kept only if it holds on almost every row where all its terms are observed and valid. Where an identity fails on the remaining <1%, section 5.2 shows that the stored value, not the identity, is wrong:

In [18]:
evidence = train.loc[~missing_pattern].copy()
evidence[non_negative_columns] = evidence[non_negative_columns].mask(evidence[non_negative_columns].lt(0))
for column in placeholder_columns:
    evidence[column] = evidence[column].mask(evidence[column].isin(placeholder_values))

identity_report = []
for rule in diag_cfg["recompute_rules"]:
    rows = evidence.query(rule["when"])
    expected = rows.eval(rule["expression"])
    observed = rows[rule["column"]]
    both = expected.notna() & observed.notna()
    holds = np.isclose(expected[both], observed[both], rtol=1e-3, atol=0.6)
    identity_report.append({
        "column": rule["column"],
        "identity": f"{rule['expression']}  (when {rule['when']})",
        "rows_checked": int(both.sum()),
        "identity_holds_%": round(100 * holds.mean(), 2),
        "NaN_it_can_fill": int((observed.isna() & expected.notna()).sum()),
    })
display(pd.DataFrame(identity_report))

,column,identity,rows_checked,identity_holds_%,NaN_it_can_fill
0,prior_mean_purchase_value,prior_total_spend / prior_purchase_count (whe...,5640,99.18,226
1,days_since_previous_purchase,customer_tenure_days (when prior_purchase_cou...,423,100.00,22
2,days_since_previous_purchase,customer_tenure_days - prior_mean_gap_days * (...,5031,100.00,179
3,prior_mean_gap_days,(customer_tenure_days - days_since_previous_pu...,5031,100.00,203
4,spend_365d,prior_total_spend (when customer_tenure_days ...,1717,99.18,68
5,prior_max_purchase_value,prior_total_spend (when prior_purchase_count ...,445,99.33,0
6,prior_min_purchase_value,prior_total_spend (when prior_purchase_count ...,445,100.00,0
7,prior_max_purchase_value,prior_total_spend - prior_min_purchase_value ...,365,99.73,0
8,prior_min_purchase_value,prior_total_spend - prior_max_purchase_value ...,365,99.73,0


**Redundant and non-informative columns** (`diagnostics.redundant_columns`, `data.drop_columns`):

In [19]:
gross_identity = np.isclose(
    evidence["gross_total"],
    evidence["net_total"] + evidence["line_discount_total"] + evidence["global_discount_total"],
    atol=0.05,
)
print(f"gross_total = net_total + line + global discount: holds for {100 * gross_identity.mean():.1f}% of rows")
print(f"corr(total_payable, net_total) = {evidence['total_payable'].corr(evidence['net_total']):.3f}")
print(f"corr(random_noise, repeat_purchase_90d) = {evidence['random_noise'].corr(evidence['repeat_purchase_90d']):.3f}")

gross_total = net_total + line + global discount: holds for 100.0% of rows
corr(total_payable, net_total) = 0.996
corr(random_noise, repeat_purchase_90d) = -0.012


- `gross_total` is an exact sum of three other columns, so a linear model gets the same information twice → dropped.
- `total_payable` and `net_total` are correlated at 0.996 → `total_payable` dropped.
- `random_noise` has no relationship with the target and, as its name says, is noise → dropped.

### 5.2 Consistency between columns

Many columns are tied to each other by accounting identities or by logic: a 30-day spend can't exceed a 90-day spend, and a mean can't exceed the maximum. `config.yaml` → `diagnostics.consistency_rules` lists every rule. Each is checked only on rows where all its terms are observed.

**First, the one identity that looked broken.** `net_total + total_tax + shipping_cost = total_payable` holds for only about 84% of rows:

In [20]:
breaks = ~np.isclose(evidence["net_total"] + evidence["total_tax"] + evidence["shipping_cost"], evidence["total_payable"], atol=0.05)
without_shipping = np.isclose(evidence["net_total"] + evidence["total_tax"], evidence["total_payable"], atol=0.05)
print(f"net + tax + shipping = total_payable: {100 * (~breaks).mean():.1f}% of rows")
print(f"net + tax            = total_payable: {100 * without_shipping.mean():.1f}% of rows")
print(f"shipping_cost > 0 where the first identity breaks: {100 * evidence.loc[breaks, 'shipping_cost'].gt(0).mean():.0f}% "
      f"| where it holds: {100 * evidence.loc[~breaks, 'shipping_cost'].gt(0).mean():.0f}%")

net + tax + shipping = total_payable: 83.7% of rows
net + tax            = total_payable: 100.0% of rows
shipping_cost > 0 where the first identity breaks: 100% | where it holds: 0%


It is not a data error. **Shipping is simply not part of `total_payable`.** `total_payable = net_total + total_tax` holds on 100% of rows, and the "broken" rows are exactly the ones with a shipping cost. This is also why `total_payable` is dropped: it is an exact sum of two other features.

**All rules:**

In [21]:
consistency_report = apply_consistency_rules(evidence.copy(), diag_cfg["consistency_rules"])
display(consistency_report[["rule", "rows_checked", "values", "column"]]
        .rename(columns={"values": "violations", "column": "set to NaN"}))

,rule,rows_checked,violations,set to NaN
0,abs(gross_total - line_discount_total - global...,6469,0,
1,abs(net_total + total_tax - total_payable) <= ...,6469,0,
2,purchase_count_30d <= purchase_count_90d,6469,0,
3,purchase_count_90d <= purchase_count_180d,6469,0,
4,purchase_count_180d <= purchase_count_365d,6469,0,
5,purchase_count_365d <= prior_purchase_count,6469,0,
6,spend_30d <= spend_90d + 0.01,6469,0,
7,spend_90d <= spend_180d + 0.01,6469,0,
8,spend_180d <= spend_365d + 0.01,6244,0,
9,prior_min_purchase_value <= prior_max_purchase...,5866,0,


Four rules are broken. **What do the violating values look like?**

In [22]:
target = config["classification"]["target"]
implied_mean = evidence["prior_total_spend"] / evidence["prior_purchase_count"]
bad_mean = (evidence["prior_purchase_count"] > 0) & evidence["prior_mean_purchase_value"].notna() & ~np.isclose(
    evidence["prior_mean_purchase_value"], implied_mean, rtol=1e-3, atol=0.6)
bad_365 = evidence["spend_365d"] > evidence["prior_total_spend"] + 0.01
bad_max = (evidence["prior_purchase_count"] > 0) & (evidence["prior_max_purchase_value"] > evidence["prior_total_spend"] + 0.01)
below_cost = evidence["cost"] > evidence["net_total"]

ratio_mean = (evidence["prior_mean_purchase_value"] / implied_mean)[bad_mean]
ratio_365 = (evidence["spend_365d"] / evidence["prior_total_spend"])[bad_365]
ratio_max = (evidence["prior_max_purchase_value"] / evidence["prior_total_spend"])[bad_max]
display(pd.DataFrame({
    "rows": [int(bad_mean.sum()), int(bad_365.sum()), int(bad_max.sum())],
    "times_too_large (min)": [ratio_mean.min(), ratio_365.min(), ratio_max.min()],
    "times_too_large (median)": [ratio_mean.median(), ratio_365.median(), ratio_max.median()],
    "times_too_large (max)": [ratio_mean.max(), ratio_365.max(), ratio_max.max()],
    "repurchase_rate": [evidence.loc[bad_mean, target].mean(), evidence.loc[bad_365, target].mean(),
                        evidence.loc[bad_max, target].mean()],
}, index=["prior_mean_purchase_value vs total / count", "spend_365d vs prior_total_spend",
          "prior_max_purchase_value vs prior_total_spend"]).round(2))
print(f"prior_mean_purchase_value above prior_max_purchase_value: {int((evidence['prior_mean_purchase_value'] > evidence['prior_max_purchase_value'] + 0.01)[bad_mean].sum())} of the {int(bad_mean.sum())}")
print(f"overall repurchase rate: {evidence[target].mean():.2f}")
print(f"sold below cost: {int(below_cost.sum())} rows, median cost / net_total = "
      f"{(evidence.loc[below_cost, 'cost'] / evidence.loc[below_cost, 'net_total']).median():.2f}")

,rows,times_too_large (min),times_too_large (median),times_too_large (max),repurchase_rate
prior_mean_purchase_value vs total / count,46,8.48,16.07,24.81,0.54
spend_365d vs prior_total_spend,39,1.53,10.81,23.18,0.56
prior_max_purchase_value vs prior_total_spend,45,1.08,4.26,22.31,0.53


prior_mean_purchase_value above prior_max_purchase_value: 45 of the 46
overall repurchase rate: 0.56
sold below cost: 39 rows, median cost / net_total = 1.22


**Decisions:**
- **`prior_mean_purchase_value` that disagrees with `prior_total_spend / prior_purchase_count`:** these values are several times too large, and almost all of them are above the customer's own maximum, which is impossible. Set to `NaN` (`invalidate`), then recomputed exactly from the identity (5.1).
- **`spend_365d` above `prior_total_spend`:** impossible, because the last 365 days are part of all prior spend. The values are again inflated, so they are set to `NaN`. For customers with ≤ 365 days of tenure the identity recovers them; the rest are imputed (median + a missing indicator, notebook 03, section 3).
- **`prior_max_purchase_value` above `prior_total_spend`:** one previous purchase can't be larger than the sum of all previous purchases. These values are inflated too, so they are set to `NaN`. No identity can recover a maximum, so they are imputed inside the pipeline (median, notebook 03, section 3). Rule added after the outlier analysis (`notebooks/04_outlier_detection.ipynb`).
- **`cost > net_total`:** these are sales below cost (a negative margin). That is unusual but possible (promotions, clearance, a commercial gesture), and unlike the other three it doesn't contradict another column, so they are **kept** (report-only rule).
- **Every other rule has 0 violations.** They stay in `config.yaml`, so new data is checked the same way.

The corrupted rows have the same repurchase rate as everybody else, so fixing them removes no real information. Section 8 measures the effect on the models.

### 5.3 Structural missingness: values that cannot exist

Missing history values are concentrated where the value is undefined: a first purchase has no previous purchase, a standard deviation needs at least two values, and a standard deviation of gaps needs at least two gaps. Missing rate (%) by number of prior purchases, on the raw training rows:

In [23]:
history_columns = [
    "days_since_previous_purchase", "prior_mean_purchase_value", "prior_min_purchase_value",
    "prior_max_purchase_value", "prior_std_purchase_value", "prior_mean_gap_days", "prior_std_gap_days",
]
history = train.loc[~missing_pattern]
prior_purchases = history["prior_purchase_count"].clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
structural_table = history[history_columns].isna().groupby(prior_purchases).mean().mul(100).round(1).T
structural_table.columns.name = "prior_purchase_count"
display(structural_table)

prior_purchase_count,0,1,2,3+
days_since_previous_purchase,100.0,4.3,5.5,3.0
prior_mean_purchase_value,100.0,4.5,3.0,3.2
prior_min_purchase_value,100.0,0.0,0.0,0.0
prior_max_purchase_value,100.0,0.0,0.0,0.0
prior_std_purchase_value,100.0,100.0,0.0,0.0
prior_mean_gap_days,100.0,100.0,5.8,3.4
prior_std_gap_days,100.0,100.0,100.0,0.0


Each column is 100% missing exactly where it is undefined. The few gaps left at 1/2/3+ prior purchases come from the corrupted values and are mostly recovered by the identities in 5.1. So these gaps are **not** MCAR/MAR/MNAR in the usual sense: they are *not applicable*. A median would invent a purchase history for a first-time customer, so instead:

- `diagnostics.structural_fill` sets them to **0** (no previous spend, no spread, no gap). This is a rule from the column definitions, not a fitted statistic.
- the flags `is_first_purchase` and `has_gap_history` let a model tell "0 because undefined" apart from a real 0.

### 5.4 Applying the rules

`prepare_table()` runs `clean_dataset()` (every rule above) on train.csv and test.csv and reports every value it changed. It then joins the customer table (notebook 02, section 11). `drop_training_rows()` is called on the training file only.

In [24]:
train_rules, cleaning_report = prepare_table(train_raw, config, clientes_raw)
test_rules, _ = prepare_table(test_raw, config, clientes_raw)
assert len(train_rules) == len(train_raw) and len(test_rules) == len(test_raw), "prepare_table must keep every row"
display(cleaning_report[["step", "column", "values"]])

train_rules, row_report = drop_training_rows(train_rules, diag_cfg, data_cfg["id_column"], config["customers"])
display(row_report)
print(f"train.csv: {len(train_raw)} -> {len(train_rules)} training rows | test.csv: {len(test_raw)} -> {len(test_rules)} rows (none dropped)")

,step,column,values
0,schema: outside valid range,days_since_previous_purchase,16
1,schema: outside valid range,prior_mean_purchase_value,16
2,schema: outside valid range,prior_mean_gap_days,16
3,schema: outside valid range,spend_365d,15
4,negative -> NaN,days_since_previous_purchase,16
5,negative -> NaN,prior_mean_purchase_value,16
6,negative -> NaN,prior_mean_gap_days,16
7,negative -> NaN,spend_365d,15
8,placeholder -> NaN,prior_mean_purchase_value,16
9,placeholder -> NaN,spend_365d,15


,rule,rows_removed
0,almost-empty rows,65
1,exact duplicates (ignoring id),0
2,repeated ids,0


train.csv: 6534 -> 6469 training rows | test.csv: 986 -> 986 rows (none dropped)


**Conclusions:** Every value the stateless rules touch is counted above:
- negatives and placeholders → `NaN`;
- 130 inflated values → `NaN` (prior mean, 365-day spend and prior maximum);
- 758 values recovered exactly from identities;
- every "not applicable" history value → 0, with a flag.

No row is removed by these rules. Only the 65 almost-empty rows leave the training file, and test.csv keeps all 986 rows.

# <font color='#2E86C1' size=6>**6. Features, targets and the locked test set**</font> <a class="anchor" id="split"></a>

[Back to TOC](#toc)

In [25]:
date_col = data_cfg["date_column"]
print(f"train.csv: {train_rules[date_col].min():%Y-%m-%d} -> {train_rules[date_col].max():%Y-%m-%d}")
print(f"test.csv:  {test_rules[date_col].min():%Y-%m-%d} -> {test_rules[date_col].max():%Y-%m-%d}")
shared = set(test_rules["customer_id"]) & set(train_rules["customer_id"])
print(f"{train_rules['customer_id'].nunique()} customers make {len(train_rules)} training purchases; "
      f"{len(shared)} of the {test_rules['customer_id'].nunique()} test customers also appear in train")

train.csv: 2015-07-11 -> 2025-02-15
test.csv:  2025-05-24 -> 2026-10-06
607 customers make 6469 training purchases; 184 of the 206 test customers also appear in train


test.csv lies **strictly after** train.csv in time, and most of its customers are already in train. A random split would let the model learn from a customer's *later* purchases and then be scored on their earlier ones, which is the opposite of what Kaggle asks. So the locked test set is the **latest 20% of training purchases** (`config.yaml` → `test_set.strategy: temporal`). It is never used to fit, tune, compare or choose anything. Development purchases from the 90 days just before it are left out (`test_set.gap_days`), because their 90-day labels are measured inside the test period.

`split_features_target()` returns `(X, y, extras)`:
- `extras` keeps `id`, `customer_id` and `purchase_date` row-aligned beside `X` (for the split and the submission file), never as model inputs;
- **both** targets are removed from `X`, so the classifier never sees `spend_next_90d` and the regressor never sees `repeat_purchase_90d`;
- `y` is `None` on test.csv: nothing downstream needs the label.

Before that, every purchase is filled from the **same customer's earlier purchases** (`fill_from_customer_history()`, train.csv + test.csv): the categories from the customer's last known value, the days since the previous purchase from the dates. This is what `prepare_task_data()` does; the evidence is in `notebooks/03_missing_values.ipynb`, section 7. Notebook 03, section 3 tested the gaps *before* this fill, which is why the categories turned out to be explained by the customer.

In [26]:
# the customer-history fill, exactly as in prepare_task_data(): past purchases only, every known purchase
history = pd.concat([train_rules, test_rules], ignore_index=True)
train_rules, _ = fill_from_customer_history(train_rules, history, diag_cfg, data_cfg["id_column"])
test_rules, _ = fill_from_customer_history(test_rules, history, diag_cfg, data_cfg["id_column"])

splits = {}
for task, task_prep in {"classification": classification_prep, "regression": regression_prep}.items():
    task_rows = task_prep.select_training_rows(train_rules, config)
    X, y, extras = task_prep.prepare_features(task_rows, config)
    X_dev, X_test, y_dev, y_test, extras_dev, extras_test = split_dev_test(X, y, extras, config["test_set"], date_col)
    splits[task] = (X_dev, X_test, y_dev, y_test, extras_dev, extras_test)
    print(f"{task:>14}: development {len(X_dev)} rows ({extras_dev[date_col].min():%Y-%m} -> {extras_dev[date_col].max():%Y-%m})"
          f" | locked test {len(X_test)} rows ({extras_test[date_col].min():%Y-%m} -> {extras_test[date_col].max():%Y-%m})")

X_kaggle, y_kaggle, extras_kaggle = classification_prep.prepare_features(test_rules, config)
print(f"{'test.csv':>14}: {len(X_kaggle)} rows, y is {y_kaggle}")

classification: development 5024 rows (2015-07 -> 2023-03) | locked test 1286 rows (2023-06 -> 2025-02)
    regression: development 2659 rows (2015-07 -> 2023-03) | locked test 685 rows (2023-06 -> 2025-02)
      test.csv: 986 rows, y is None


**Conclusions:** Each task has a development set (to learn from and compare on) and a locked test set made of the latest purchases, separated by a 90-day gap. The regression uses only repeat purchasers with an observed spend. test.csv goes through the same stateless steps, and `y` is `None` for it.

# <font color='#2E86C1' size=6>**7. Temporal cross-validation and the decision rule**</font> <a class="anchor" id="cv"></a>

Every later choice is made on the **development set** of section 6 (the locked test set plays no part), with one protocol that every notebook from here on reuses:

- **5 expanding-window temporal folds** (`config.yaml` → `cv`). Each fold trains on the past and validates on the next block of purchases, like test.csv. Training rows from the 90 days just before a validation block are left out (`cv.gap_days`), because their 90-day labels are measured inside that block.
- **The same folds for every option**, so two options can be compared fold by fold (`paired_comparison()`): the mean difference, its standard error, and the number of folds where each option is better.
- **One reference pipeline per task:** the recipe and the model in `config.yaml` (`<task>.preprocessing`, `<task>.model`), chosen in notebook 05. An option is tested by changing only that option.
- **The decision rule:** keep the current choice unless another one is **decisively** better (a mean advantage of more than 2 standard errors), or has a clear reason that most folds confirm. Switching on noise would make the pipeline change every time the data changes. Corrections of impossible or undefined values are kept for correctness, even without a gain (section 8).

The temporal folds of each task:

[Back to TOC](#toc)

In [27]:
TASK_MODELS = {"classification": classification_model, "regression": regression_model}

task_folds = {}
for task, (X_dev, X_test, y_dev, y_test, extras_dev, extras_test) in splits.items():
    task_folds[task] = temporal_folds(extras_dev[date_col], config["cv"]["n_splits"], config["cv"]["gap_days"])
    display(pd.DataFrame([
        {"task": task, "fold": k,
         "train_rows": len(tr), "train_until": f"{extras_dev[date_col].iloc[tr].max():%Y-%m-%d}",
         "validation_rows": len(va), "validation_from": f"{extras_dev[date_col].iloc[va].min():%Y-%m-%d}",
         "validation_until": f"{extras_dev[date_col].iloc[va].max():%Y-%m-%d}"}
        for k, (tr, va) in enumerate(task_folds[task])
    ]))


,task,fold,train_rows,train_until,validation_rows,validation_from,validation_until
0,classification,0,750,2016-12-25,838,2017-03-26,2018-05-19
1,classification,1,1516,2018-02-23,837,2018-05-25,2019-08-24
2,classification,2,2347,2019-05-25,837,2019-08-26,2020-11-14
3,classification,3,3185,2020-08-02,837,2020-11-15,2021-11-30
4,classification,4,3936,2021-08-31,837,2021-11-30,2023-03-18


,task,fold,train_rows,train_until,validation_rows,validation_from,validation_until
0,regression,0,383,2017-01-30,443,2017-05-03,2018-06-01
1,regression,1,817,2018-03-03,443,2018-06-03,2019-09-07
2,regression,2,1227,2019-06-08,443,2019-09-07,2020-11-02
3,regression,3,1675,2020-08-02,443,2020-11-02,2021-11-13
4,regression,4,2096,2021-08-14,443,2021-11-13,2023-03-21


# <font color='#2E86C1' size=6>**8. Consistency rules in the models**</font> <a class="anchor" id="consistency-models"></a>

The consistency rules of section 5.2 set 130 impossible values to `NaN`. Each task's reference pipeline (section 7) is scored **with and without** them, on the same temporal folds, and compared fold by fold.

[Back to TOC](#toc)

In [28]:
no_consistency = copy.deepcopy(config)
no_consistency["diagnostics"]["consistency_rules"] = []

rows = []
for task in ["classification", "regression"]:
    recipe = task_preprocessing_config(config, task)
    metric = config[task]["selection_metric"]
    scores = {}
    for name, cfg in {"with": config, "without": no_consistency}.items():
        data = prepare_task_data(cfg, task)
        assert data["X_dev"].index.equals(splits[task][0].index)       # same rows -> same folds
        pipe = build_pipeline(recipe, TASK_MODELS[task].build_model(config[task]["model"]))
        scores[name] = cross_validate_pipeline(pipe, data["X_dev"], data["y_dev"], task_folds[task], SCORING[task])[f"validation_{metric}"].to_numpy()
    rows.append({"task": task, "metric": metric, "with rules": scores["with"].mean(), "without rules": scores["without"].mean(),
                 **paired_comparison(scores["with"], scores["without"], metric != "rmse")})
display(pd.DataFrame(rows).round(4))

,task,metric,with rules,without rules,mean_advantage,se,folds_a_better,n_folds,decisive
0,classification,accuracy,0.7078,0.7133,-0.0055,0.0023,0,5,True
1,regression,rmse,1942.4794,1937.8509,-4.6286,12.7102,2,5,False


**Verdict: keep the rules.** For regression they make no decisive difference (4.6 ± 12.7 RMSE in favour of the uncorrected values, better on 2 of 5 folds). For classification, fixing the 130 impossible values costs 0.006 accuracy, which is decisive (worse on all 5 folds). We keep them anyway:
- **They carry no repurchase information.** The corrupted rows have the same repurchase rate as everyone else (0.54, 0.56 and 0.53 vs 0.56, section 5.2).
- **The loss most likely comes from scaling.** Values up to 25× too large stretch the scale of three columns for every other row, and logistic regression happens to score slightly better on that distorted scale.
- **Robustness.** A model that depends on impossible values would behave unpredictably on new data, where the corruption could be different or absent.

Correct inputs come first; tuning (later) can recover accuracy honestly.

# <font color='#2E86C1' size=6>**9. From this notebook to the pipeline**</font> <a class="anchor" id="to-pipeline"></a>

| This notebook | Pipeline |
|---|---|
| Section 1, 2.2 | `src/data.py` → `load_data(path, placeholder_tokens)`, `report_placeholders()`, `standardize_columns()`; `config.yaml` → `data`, `diagnostics.placeholder_tokens`, `diagnostics.column_renames` |
| Section 2.8 | `src/data.py` → `validate_schema()`, `standardize_dtypes()`; `config.yaml` → `schema`, `dtypes` |
| Section 4.4 | `src/data.py` → `validate_customer_ids()` |
| Sections 2.2, 5.1–5.4 (customer join: notebook 02, section 11) | `src/cleaning.py` → `prepare_table()` = `validate_schema()` + `clean_dataset()` (`flag_invalid_values`, `apply_consistency_rules`, `apply_recompute_rules`, `apply_structural_fill`, `add_flags`, `add_date_features`, `add_derived_features`, redundant columns) + `add_customer_features()` + `standardize_dtypes()`; `config.yaml` → `diagnostics`, `customers` |
| Sections 2.1, 2.4, 4 | `src/cleaning.py` → `drop_training_rows()`: training file only, before the split |
| Section 6 | `src/preprocessing.py` → `split_features_target()`, `split_dev_test()`; `src/pipeline.py` → `prepare_task_data()`; `config.yaml` → `data`, `test_set` |
| Sections 7–8 | `src/evaluate.py` → `temporal_folds()`, `cross_validate_pipeline()`, `paired_comparison()`; `src/pipeline.py` → `prepare_task_data()`; `config.yaml` → `cv`, `diagnostics.consistency_rules` |

[Back to TOC](#toc)